In [11]:
import re
import pandas as pd
import joblib
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline

drive.mount("/content/gdrive")

%cd "/content/gdrive/MyDrive/datasets"

# columns has the name of each column.
df_raw = pd.read_csv('cpp_dataset.csv', delimiter=',', encoding='latin1')
df_raw.head()

df_cpp = df_raw.copy()


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive/datasets


In [12]:
CPP_KEYWORDS = {
    "int", "float", "double", "char", "bool", "void", "if", "else", "for",
    "while", "return", "class", "struct", "public", "private", "protected",
    "namespace", "using", "switch", "case", "break", "continue", "auto"
}

CPP_TYPES = {
    "int", "float", "double", "char", "bool", "void", "auto"
}

SYMBOLS = {
    "(": "LPAREN", ")": "RPAREN",
    "{": "LBRACE", "}": "RBRACE",
    ";": "SEMICOLON", ",": "COMMA"
}

TOKEN_REGEX = re.compile(
    r"""
    (?P<COMMENT>//[^\n]*|/\*[\s\S]*?\*/) |
    (?P<STRING>"(?:\\.|[^"\\])*") |
    (?P<NUMBER>\b\d+\b) |
    (?P<OPERATOR>[+\-*/%=<>!&|]+) |
    (?P<SYMBOL>[()\{\};,]) |
    (?P<IDENTIFIER>\b[A-Za-z_][A-Za-z0-9_]*\b) |
    (?P<WHITESPACE>\s+)
    """,
    re.VERBOSE
)


def tokenize_cpp(code):
    tokens = []
    previous_token = None

    for match in TOKEN_REGEX.finditer(code):
        kind = match.lastgroup
        value = match.group()

        if kind == "WHITESPACE":
            continue

        if kind == "IDENTIFIER":
            if value in CPP_KEYWORDS:
                if value in CPP_TYPES:
                    token = f"TYPE:{value}"
                else:
                    token = f"KEYWORD:{value}"
            else:
                token = "IDENTIFIER"

        elif kind == "OPERATOR":
            token = f"OP:{value}"

        elif kind == "SYMBOL":
            token = SYMBOLS[value]

        elif kind == "NUMBER":
            token = "NUMBER"

        elif kind == "STRING":
            token = "STRING"

        elif kind == "COMMENT":
            token = "COMMENT"

        else:
            token = "UNKNOWN"

        tokens.append(token)
        previous_token = token

    return tokens

In [13]:
df_cpp["tokens"] = df_cpp["code"].apply(tokenize_cpp)
df_cpp["tokens_text"] = df_cpp["tokens"].apply(lambda x: " ".join(x))

df_cpp[["code", "tokens_text"]].head()

,code,tokens_text
0,#include <iostream>\n#include <vector>\n#inclu...,IDENTIFIER OP:< IDENTIFIER OP:> IDENTIFIER OP:...
1,#include <iostream>\n#include <string>\n\nusin...,IDENTIFIER OP:< IDENTIFIER OP:> IDENTIFIER OP:...
2,#include <iostream>\nusing namespace std;\n\ni...,IDENTIFIER OP:< IDENTIFIER OP:> KEYWORD:using ...
3,/*\n * Arithmetic-Geometric Mean of 1 & 1/sqrt...,COMMENT IDENTIFIER OP:< IDENTIFIER OP:> IDENTI...
4,#include <iostream>\n#include <string>\n#inclu...,IDENTIFIER OP:< IDENTIFIER OP:> IDENTIFIER OP:...


In [14]:
X = df_cpp["tokens_text"]
y = df_cpp["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 3)
    )),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 3))),
                ('rf',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, n_jobs=-1,
                                        random_state=42))])

In [16]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8845598845598845

Matriz de confusión:
[[  40  148]
 [  12 1186]]

Reporte:
              precision    recall  f1-score   support

           0       0.77      0.21      0.33       188
           1       0.89      0.99      0.94      1198

    accuracy                           0.88      1386
   macro avg       0.83      0.60      0.64      1386
weighted avg       0.87      0.88      0.85      1386

